# VSD Left/Right Rename Executor

Performs the Left↔Right swap for **mislabeled VSD cases only**, across the paired chain, renaming
folders + files and updating the metadata CSV key columns. Driven by
`reports/figures/vsd_lr_validation/vsd_lr_verdicts.csv` (rows with `verdict == MISLABELED`).

### How to use — DESTRUCTIVE, so this is a two-step, gated flow
1. **Run cells 1–3.** Cell 3 (yes, cell index 2 — "cell 3" counting the header) prints the **exact
   folders** that will be renamed and saves the **full manifest** (folders + files + CSV-cell edits)
   to `reports/figures/vsd_lr_validation/vsd_lr_rename_manifest.csv` for easy review in Excel.
   **Nothing is renamed yet.**
2. **Review that manifest** (on screen or in the CSV). If it looks right, set **`CONFIRM = True`** in
   the config cell and re-run cell 4 to apply. Re-running the swap again reverts it (Left↔Right is
   self-inverse); the `(LR might be wrong)` strip is one-way. An undo manifest + a marker file are
   written.

**Scope guard:** only the 4 approved roots below are ever touched, and only keys `VSD_*` with a
`MISLABELED` verdict. Any path outside the allowlist aborts the run.

Run with `./.venv/Scripts/python.exe`.

In [1]:
# ==== Config ====
import re, json, shutil
from datetime import datetime
from pathlib import Path
import pandas as pd

CONFIRM = True              # <-- set True ONLY after reviewing the cell-3 manifest
STRIP_PARENTHETICAL = True   # strip "(LR might be wrong)" from STL case folders

def find_root(start: Path) -> Path:
    for c in [start.resolve(), *start.resolve().parents]:
        if (c / "data/interim/gt_per_bone_256/healthy").exists():
            return c
    raise FileNotFoundError("project root not found")
ROOT = find_root(Path.cwd())

GT_DIR     = ROOT / "data/interim/gt_per_bone_256/healthy"
RAW_DIR    = ROOT / "data/raw/healthy"
UPRIGHT    = ROOT / "data/external/upright/VSD_cases"
PREDRR     = ROOT / "data/interim/predrr/healthy"
OCC_DIR    = ROOT / "data/interim/predrr_occupancy_256"
STL_ROOT   = ROOT / "data/external/ground_truth/VSD_ground_truth"
CSV_FILES  = [GT_DIR.parent / n for n in
              ("gt_per_bone_metadata.csv", "gt_per_bone_alignment.csv", "gt_per_bone_tsdf_qc.csv")]
CSV_FILES  = [c for c in CSV_FILES if c.exists()]
# every path we touch must live under one of these roots:
ALLOWLIST  = [GT_DIR, RAW_DIR, UPRIGHT, PREDRR, OCC_DIR, STL_ROOT] + CSV_FILES
MARKER     = ROOT / "data/interim/vsd_lr_rename_applied.json"
UNDO_CSV   = ROOT / "data/interim/vsd_lr_rename_undo.csv"
MANIFEST_CSV = ROOT / "reports/figures/vsd_lr_validation/vsd_lr_rename_manifest.csv"

# mislabeled VSD subjects from the verdict CSV
verdicts = pd.read_csv(ROOT / "reports/figures/vsd_lr_validation/vsd_lr_verdicts.csv")
mis = verdicts[(verdicts.verdict == "MISLABELED") & (verdicts.key.str.startswith("VSD_"))]
BASES = sorted({re.match(r"(VSD_(z?\d+))_", k).group(1) for k in mis.key})
print(f"ROOT: {ROOT}")
print(f"Mislabeled subjects: {len(BASES)}  (from {len(mis)} case-sides)")
print("CONFIRM =", CONFIRM, "| STRIP_PARENTHETICAL =", STRIP_PARENTHETICAL)

ROOT: C:\Users\Chan Zheng Shao\OneDrive\Desktop\Github Repo\TestProject\TestProject
Mislabeled subjects: 30  (from 58 case-sides)
CONFIRM = True | STRIP_PARENTHETICAL = True


In [2]:
# ==== Build the net rename plan + safety checks (DRY RUN — nothing renamed) ====
def key_swap(s: str, base: str) -> str:
    L, R, PH = f"{base}_Left", f"{base}_Right", "\x00"
    return s.replace(L, PH).replace(R, L).replace(PH, R)

def under_allowlist(p: Path) -> bool:
    rp = p.resolve()
    return any(rp == a.resolve() or a.resolve() in rp.parents for a in ALLOWLIST)

def stl_case_dir(base):
    for p in STL_ROOT.iterdir():
        if p.is_dir() and re.match(r"(VSD_\w+)", p.name).group(1) == base:
            return p
    return None

def stl_target(p: Path, base: str) -> Path:
    parts = list(p.relative_to(STL_ROOT).parts)
    parts[0] = re.match(r"(VSD_\w+)", parts[0]).group(1)          # strip parenthetical
    if len(parts) > 1 and parts[1] in ("Left", "Right"):
        parts[1] = "Right" if parts[1] == "Left" else "Left"     # swap side subfolder
    parts = [key_swap(x, base) for x in parts]                    # swap keyed filename token
    return STL_ROOT.joinpath(*parts)

folder_ops, file_ops = [], []          # (src, dst)
for base in BASES:
    # --- flat-file locations: predrr, occupancy, upright, raw/healthy/VSD.<id> ---
    raw_sub = RAW_DIR / base.replace("VSD_", "VSD.")
    for loc in [PREDRR, OCC_DIR, UPRIGHT, raw_sub]:
        if loc.exists():
            for f in list(loc.glob(f"*{base}_Left*")) + list(loc.glob(f"*{base}_Right*")):
                if f.is_file():
                    file_ops.append((f, f.with_name(key_swap(f.name, base))))
    # --- GT: KEY folders + nested files + spotcheck pngs ---
    for side in ("Left", "Right"):
        d = GT_DIR / f"{base}_{side}"
        if d.is_dir():
            folder_ops.append((d, GT_DIR / key_swap(d.name, base)))
            for f in d.iterdir():
                if f.is_file():
                    file_ops.append((f, (GT_DIR / key_swap(d.name, base)) / key_swap(f.name, base)))
    for png in GT_DIR.glob(f"_spotcheck_{base}_*"):
        file_ops.append((png, png.with_name(key_swap(png.name, base))))
    # --- STL: case dir (paren strip) + side subfolders + files ---
    cdir = stl_case_dir(base)
    if cdir:
        if STRIP_PARENTHETICAL and cdir.name != base:
            folder_ops.append((cdir, STL_ROOT / base))
        for sub in ("Left", "Right"):
            sd = cdir / sub
            if sd.is_dir():
                folder_ops.append((sd, stl_target(sd, base)))
                for f in sd.iterdir():
                    if f.is_file():
                        file_ops.append((f, stl_target(f, base)))

# CSV cell edits (row-level detail, for the manifest export + the summary count)
csv_edit_rows = []
for csv in CSV_FILES:
    c = pd.read_csv(csv)
    for col in [x for x in ("key", "source_stl") if x in c.columns]:
        for idx, val in c[col].items():
            sval = str(val)
            for base in BASES:
                if re.search(rf"{re.escape(base)}_(Left|Right)", sval):
                    csv_edit_rows.append({
                        "kind": "csv_cell", "base": base,
                        "location": str(csv.relative_to(ROOT)),
                        "current": f"row {idx}, col '{col}': {sval}",
                        "proposed": f"row {idx}, col '{col}': {key_swap(sval, base)}",
                    })
                    break
csv_hits = len(csv_edit_rows)

# ---- SAFETY CHECKS ----
problems = []
for src, dst in folder_ops + file_ops:
    if not under_allowlist(src) or not under_allowlist(dst):
        problems.append(f"OUTSIDE ALLOWLIST: {src} -> {dst}")
assert not problems, "\n".join(problems)

def base_of(p: Path) -> str:
    m = re.search(r"VSD_(z?\d+)", p.name) or re.search(r"VSD_(z?\d+)", str(p))
    return f"VSD_{m.group(1)}" if m else ""

# ---- Human-readable manifest -> reports/ ----
manifest_rows = []
for src, dst in folder_ops:
    manifest_rows.append({"kind": "folder", "base": base_of(src),
                          "location": str(src.parent.relative_to(ROOT)),
                          "current": src.relative_to(ROOT).as_posix(),
                          "proposed": dst.relative_to(ROOT).as_posix()})
for src, dst in file_ops:
    manifest_rows.append({"kind": "file", "base": base_of(src),
                          "location": str(src.parent.relative_to(ROOT)),
                          "current": src.relative_to(ROOT).as_posix(),
                          "proposed": dst.relative_to(ROOT).as_posix()})
manifest_rows += csv_edit_rows

manifest_df = pd.DataFrame(manifest_rows,
                           columns=["kind", "base", "location", "current", "proposed"])
manifest_df = manifest_df.sort_values(["base", "kind", "location", "current"]).reset_index(drop=True)
MANIFEST_CSV.parent.mkdir(parents=True, exist_ok=True)
manifest_df.to_csv(MANIFEST_CSV, index=False)

print("=== EXACT FOLDERS TO BE RENAMED (review these) ===")
for src, dst in folder_ops:
    print(f"  {src.relative_to(ROOT)}")
    print(f"      -> {dst.relative_to(ROOT)}")
print(f"\nFolder renames: {len(folder_ops)} | file renames: {len(file_ops)} | "
      f"CSV cell edits: {csv_hits}")
print(f"Allowlist check: PASS ({len(folder_ops)+len(file_ops)} paths, all inside approved roots)")
print(f"\nFull readable manifest saved to: {MANIFEST_CSV.relative_to(ROOT)}")
print(f"  ({len(manifest_df)} rows: kind, base, location, current, proposed — open in Excel to review)")
print("\nDRY RUN — nothing renamed. Review the manifest above/CSV, then set CONFIRM=True and run cell 4.")

=== EXACT FOLDERS TO BE RENAMED (review these) ===
  data\interim\gt_per_bone_256\healthy\VSD_001_Left
      -> data\interim\gt_per_bone_256\healthy\VSD_001_Right
  data\interim\gt_per_bone_256\healthy\VSD_001_Right
      -> data\interim\gt_per_bone_256\healthy\VSD_001_Left
  data\external\ground_truth\VSD_ground_truth\VSD_001\Left
      -> data\external\ground_truth\VSD_ground_truth\VSD_001\Right
  data\external\ground_truth\VSD_ground_truth\VSD_001\Right
      -> data\external\ground_truth\VSD_ground_truth\VSD_001\Left
  data\interim\gt_per_bone_256\healthy\VSD_002_Left
      -> data\interim\gt_per_bone_256\healthy\VSD_002_Right
  data\interim\gt_per_bone_256\healthy\VSD_002_Right
      -> data\interim\gt_per_bone_256\healthy\VSD_002_Left
  data\external\ground_truth\VSD_ground_truth\VSD_002\Left
      -> data\external\ground_truth\VSD_ground_truth\VSD_002\Right
  data\external\ground_truth\VSD_ground_truth\VSD_002\Right
      -> data\external\ground_truth\VSD_ground_truth\VSD_002\Le

In [3]:
# ==== APPLY (guarded) ====
def do_pair_swap(a: Path, b: Path):
    ae, be = a.exists(), b.exists()
    if ae and be:
        tmp = a.with_name(a.name + ".__swaptmp__")
        assert not tmp.exists()
        a.rename(tmp); b.rename(a); tmp.rename(b)
    elif ae:
        assert not b.exists(); a.rename(b)
    elif be:
        assert not a.exists(); b.rename(a)

def flip_nested(d: Path, base: str):
    for f in list(d.iterdir()):
        if f.is_file() and (f"{base}_Left" in f.name or f"{base}_Right" in f.name):
            f.rename(f.with_name(key_swap(f.name, base)))

if not CONFIRM:
    print("CONFIRM is False — not applying. Review cell 3, set CONFIRM=True, re-run this cell.")
else:
    assert not MARKER.exists(), (f"{MARKER} exists — swap already applied. Delete it only if you "
                                 "intend to run again (which REVERTS the swap).")
    applied = []
    # 1) STL parenthetical strip
    for base in BASES:
        cdir = stl_case_dir(base)
        if STRIP_PARENTHETICAL and cdir and cdir.name != base:
            tgt = STL_ROOT / base; assert not tgt.exists()
            cdir.rename(tgt); applied.append((str(cdir), str(tgt)))
    # 2) GT KEY dir swaps + nested filename flip
    for base in BASES:
        a, b = GT_DIR / f"{base}_Left", GT_DIR / f"{base}_Right"
        do_pair_swap(a, b)
        for d in (a, b):
            if d.is_dir(): flip_nested(d, base)
        applied.append((str(a), f"swap<->{b.name}"))
    # 3) STL side-subfolder swaps + nested filename flip
    for base in BASES:
        cdir = STL_ROOT / base
        if cdir.exists():
            a, b = cdir / "Left", cdir / "Right"
            do_pair_swap(a, b)
            for d in (a, b):
                if d.is_dir(): flip_nested(d, base)
            applied.append((str(a), f"swap<->{b.name}"))
    # 4) flat-file swaps (predrr, occupancy, upright, raw)
    for base in BASES:
        raw_sub = RAW_DIR / base.replace("VSD_", "VSD.")
        for loc in [PREDRR, OCC_DIR, UPRIGHT, raw_sub]:
            if not loc.exists(): continue
            names = {f.name for f in loc.iterdir() if f.is_file()
                     and (f"{base}_Left" in f.name or f"{base}_Right" in f.name)}
            done = set()
            for nm in names:
                if nm in done: continue
                swpd = key_swap(nm, base)
                do_pair_swap(loc / nm, loc / swpd)
                done.add(nm); done.add(swpd)
                applied.append((str(loc / nm), f"swap<->{swpd}"))
        # spotcheck pngs (flat, in GT_DIR)
        pngs = {p.name for p in GT_DIR.glob(f"_spotcheck_{base}_*")}
        done = set()
        for nm in pngs:
            if nm in done: continue
            swpd = key_swap(nm, base)
            do_pair_swap(GT_DIR / nm, GT_DIR / swpd)
            done.add(nm); done.add(swpd); applied.append((str(GT_DIR / nm), f"swap<->{swpd}"))
    # 5) CSV key-column token swap
    for csv in CSV_FILES:
        c = pd.read_csv(csv); changed = False
        for col in [x for x in ("key", "source_stl") if x in c.columns]:
            def fix(v):
                s = str(v)
                for base in BASES: s = key_swap(s, base)
                return s
            new = c[col].map(fix)
            if not new.equals(c[col].astype(str)): c[col] = new; changed = True
        if changed:
            c.to_csv(csv, index=False); applied.append((str(csv), "csv key columns swapped"))
    # 6) undo manifest + marker
    pd.DataFrame(applied, columns=["src", "op"]).to_csv(UNDO_CSV, index=False)
    MARKER.write_text(json.dumps({"applied_at": datetime.now().isoformat(),
                                  "subjects": BASES, "n_ops": len(applied)}, indent=2))
    print(f"APPLIED {len(applied)} operations. Undo manifest: {UNDO_CSV}")
    print("Re-run vsd_lr_label_validation.ipynb to confirm all cases now read OK.")

APPLIED 155 operations. Undo manifest: C:\Users\Chan Zheng Shao\OneDrive\Desktop\Github Repo\TestProject\TestProject\data\interim\vsd_lr_rename_undo.csv
Re-run vsd_lr_label_validation.ipynb to confirm all cases now read OK.
